In [ ]:
import csv
import os
import time
import requests
import re
import psycopg2
from psycopg2 import sql
from datetime import datetime, timezone
from typing import Optional
from db_operations import get_db_conn
from dotenv import load_dotenv

load_dotenv("../src/config/.env")

GITHUB_PAT = os.getenv("GITHUB_TOKEN")

# Input and Output file names
INPUT_CSV_FILE = '../results/apple_ml-pointersect_matches.csv'
OUTPUT_CSV_FILE = '../results/processed_data/4Granular.csv'


API_CALL_DELAY_SECONDS = 1.2

 
# Granular 1
def github_repo_exists(repo_url: str) -> bool:
    try:
        match = re.search(r"github\.com/([^/]+)/([^/]+)", repo_url)
        if not match:
            return False
        owner = match.group(1)
        repo = match.group(2)
        base_repo_url = f"https://github.com/{owner}/{repo}"
        response = requests.head(base_repo_url, allow_redirects=True)
        return response.status_code == 200
    except requests.exceptions.RequestException:
        return False

# Granular 2
def check_version_and_get_sha(repo_url, version):
    match = re.search(r"github\.com/([^/]+)/([^/]+)", repo_url)
    if not match:
        print(f"--> Invalid GitHub URL: {repo_url}")
        return None, None, None
    owner, repo = match.groups()

    try:
        timestamp_sec = int(version) / 1000
        dt_object = datetime.fromtimestamp(timestamp_sec, tz=timezone.utc)
        iso_timestamp = dt_object.strftime('%Y-%m-%dT%H:%M:%SZ')
    except (ValueError, TypeError):
        print(f"--> Invalid timestamp format: {version}")
        return None, None, None

    api_url = f"https://api.github.com/repos/{owner}/{repo}/commits"
    headers = {'Authorization': f'token {GITHUB_PAT}', 'Accept': 'application/vnd.github.v3+json'}
    params = {'until': iso_timestamp, 'per_page': 1}
    
    time.sleep(API_CALL_DELAY_SECONDS)

    try:
        response = requests.get(api_url, headers=headers, params=params)
        if response.status_code in [404, 409, 403]: # Added 403 for PAT issues
            print(f"--> Repo not found, empty, private, or PAT invalid: {owner}/{repo} (Status: {response.status_code})")
            return None, None, None
        
        response.raise_for_status()
        commits = response.json()
        
        if commits:
            commit_sha = commits[0]['sha']
            print(f"--> Found version for timestamp {version} in {owner}/{repo}. Commit SHA: {commit_sha}")
            return commit_sha, owner, repo
        else:
            print(f"--> No version found for timestamp {version} in {owner}/{repo}")
            return None, None, None

    except requests.exceptions.RequestException as e:
        print(f"--> Request failed for {owner}/{repo}: {e}")
        return None, None, None

#Granular 3
def find_file_at_exact_path(full_file_url: str) -> bool:
    url_without_fragment = full_file_url.split('#')[0]
    try:
        response = requests.head(url_without_fragment, allow_redirects=True, timeout=10)
        return response.status_code == 200

    except requests.exceptions.RequestException:
        return False



#Granular 4
def find_method_in_file(url: str, method_name: str) -> Optional[str]:
    try:
        # 1. Convert URL to raw format and fetch content
        base_url = url.split('#')[0]
        raw_url = base_url.replace('github.com', 'raw.githubusercontent.com').replace('/blob/', '/').replace('/./', '/')

        
        response = requests.get(raw_url, timeout=10)
        response.raise_for_status()  # Check for HTTP errors (like 404 Not Found)

        # 2. Search for the method in the content
        for line in response.text.splitlines():
            if method_name in line:
                return line  

    except requests.exceptions.RequestException as e:
        # This block handles network, timeout, or HTTP errors
        print(f"Failed to process '{url}'. Reason: {e}")
        return None  # Return None on any exception

    # 3. If the loop completes, the method wasn't found
    return None


In [8]:
def import_csv_to_table(csv_file, table_name, drop_if_exists=True):
    """Read CSV and insert data into PostgreSQL table."""
    conn = get_db_conn()
    cur = conn.cursor()

    # Drop table if needed
    if drop_if_exists:
        cur.execute(sql.SQL("DROP TABLE IF EXISTS {}").format(sql.Identifier(table_name)))

    # Create table
    create_table_query = sql.SQL("""
        CREATE TABLE IF NOT EXISTS {table} (
            hash TEXT,
            project_id BIGINT,
            version BIGINT,
            license TEXT,
            method_name TEXT,
            file_location TEXT,
            repository_url TEXT,
            language TEXT,
            granular_level_reached INT,
            raw_url TEXT,
            line_content TEXT,
            target_line_number TEXT
        )
    """).format(table=sql.Identifier(table_name))
    cur.execute(create_table_query)

    # Read CSV and insert
    with open(csv_file, 'r', newline='', encoding='utf-8') as f:
        reader = csv.reader(f)
        header = next(reader)
        header += ['granular_level_reached', 'raw_url', 'line_content', 'target_line_number']
        
        insert_columns = [
            "hash", "project_id", "version", "license", "method_name",
            "file_location", "repository_url", "language",
            "granular_level_reached", "raw_url", "line_content", "target_line_number"
        ]
        placeholders = sql.SQL(", ").join([sql.Placeholder() for _ in insert_columns])
        insert_query = sql.SQL("""
            INSERT INTO {table} ({fields})
            VALUES ({values})
        """).format(
            table=sql.Identifier(table_name),
            fields=sql.SQL(", ").join(map(sql.Identifier, insert_columns)),
            values=placeholders
        )

        for row in reader:
            cur.execute(insert_query, row + [0, "", "", ""])  # default placeholders

    conn.commit()
    cur.close()
    conn.close()
    print(f"CSV imported into table '{table_name}' successfully.")


In [9]:
def update_granular_levels(TABLE_NAME):
    conn = get_db_conn()
    cur = conn.cursor()

    # --- Fetch rows from table ---
    cur.execute(f"SELECT hash, project_id, version, license, method_name, file_location, repository_url, language FROM {TABLE_NAME}")
    rows = cur.fetchall()

    for i, row in enumerate(rows, 1):
        try:
            print(f"Processing row {i}...")
            hash_val, project_id, version, license_, method_name, file_location, repo_url, language = row

            # Default values
            granular_level_reached = 0
            raw_url = ""
            line_content = ""
            target_line_number = ""

            # Verification steps
            if github_repo_exists(repo_url):
                granular_level_reached = 1
                commit_sha, _, _ = check_version_and_get_sha(repo_url, version)
                if commit_sha:
                    granular_level_reached = 2
                    if find_file_at_exact_path(repo_url):
                        granular_level_reached = 3
                        if find_method_in_file(repo_url, method_name):
                            granular_level_reached = 4

            # --- Update DB row ---
            update_query = f"""
                UPDATE {TABLE_NAME}
                SET granular_level_reached = %s,
                    raw_url = %s,
                    line_content = %s,
                    target_line_number = %s
                WHERE hash = %s AND project_id = %s AND version = %s
            """
            cur.execute(update_query, (granular_level_reached, raw_url, line_content, target_line_number,
                                       hash_val, project_id, version))
        except Exception as e:
            print(f"Error processing row {i}: {e}")
            continue

    conn.commit()
    cur.close()
    conn.close()
    print("Granular levels updated for all rows.")

In [10]:
INPUT_CSV_FILE = '../results/data/verifier_demo.csv'
TABLE_NAME = "repository_data_source_verifier"  # change if needed
DROP_IF_EXISTS = True

def main():
    import_csv_to_table('../results/data/verifier_demo.csv', TABLE_NAME, DROP_IF_EXISTS)
    update_granular_levels(TABLE_NAME)

In [11]:
if __name__ == '__main__':
    main()

CSV imported into table 'repository_data_source_verifier' successfully.
Processing row 1...
--> Found version for timestamp 1756905068000 in IBM/terratorch. Commit SHA: 821e80b6407af2cb4eb70ccc689b53915a1fde45
Processing row 2...
--> Found version for timestamp 1756894106000 in IBM/unitxt. Commit SHA: fa5b604db4c49142adc276879be90388840b8d24
Processing row 3...
--> Found version for timestamp 1756849791000 in IBM/DeepRest. Commit SHA: 9c368eeef0218232f08f0c498a43cd892232277a
Processing row 4...
--> Found version for timestamp 1756849791000 in IBM/DeepRest. Commit SHA: 9c368eeef0218232f08f0c498a43cd892232277a
Processing row 5...
--> Found version for timestamp 1756849791000 in IBM/DeepRest. Commit SHA: 9c368eeef0218232f08f0c498a43cd892232277a
Processing row 6...
--> Found version for timestamp 1756849791000 in IBM/DeepRest. Commit SHA: 9c368eeef0218232f08f0c498a43cd892232277a
Processing row 7...
--> Found version for timestamp 1756849791000 in IBM/DeepRest. Commit SHA: 9c368eeef0218232f

In [ ]:
'''
\copy (SELECT * FROM repository_data LIMIT 10) TO '/tmp/validation_mined_data.csv' WITH CSV HEADER;

sudo mv /tmp/validation_mined_data.csv /searchSECO-miner/results/data
'''

''